In [139]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
import validation
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import make_pipeline



In [140]:
to_predict_mens = pd.read_csv("to_predict_mens.csv")

### Prepare data 

In [141]:
def prepare_data(to_predict_mens, season):

    to_predict_mens_first_round_train = to_predict_mens[(to_predict_mens["GameRound"] == 1)
                                                        & (to_predict_mens.final_odds.notnull())
                                                        & (to_predict_mens.Season < season)
                                                        ].copy()
    
    to_predict_mens_train = to_predict_mens[(to_predict_mens.Season < season)].copy()

    to_predict_mens_first_round_test = to_predict_mens[(to_predict_mens.Season == season)
                                                    & (to_predict_mens.GameRound == 1)
                                                    & (to_predict_mens.final_odds.notnull())
                                                    ].copy()

    to_predict_mens_other_rounds_test = to_predict_mens[(to_predict_mens.Season == season)
                                                    & (to_predict_mens.GameRound > 1)
                                                    ].copy()
    
    to_predict_mens_missing_games = to_predict_mens[
                                                 ( 
                                                     (to_predict_mens.GameRound.isna())
                                                     | (to_predict_mens.GameRound == 0)
                                                     )
                                                & (to_predict_mens.type == "Prediction")
                                                ]
    
    return to_predict_mens_first_round_train, to_predict_mens_train, to_predict_mens_first_round_test, to_predict_mens_other_rounds_test, to_predict_mens_missing_games
    


### Odds Model Training

In [142]:
def train_odds_model(to_predict_mens_first_round_train):

    best_params = {"C": .1}
    model = LogisticRegression(**best_params)
    pipeline = make_pipeline(StandardScaler(), model)
    odds_model = pipeline.fit(to_predict_mens_first_round_train[["final_odds"]], to_predict_mens_first_round_train["Outcome"])
    
    return odds_model


### Statistics Model Training

In [143]:
def train_statistics_model(to_predict_mens_train, statistics_features):

    best_params = {"C": .1}
    model = LogisticRegression(**best_params)
    pipeline = make_pipeline(StandardScaler(), model)
    statistics_model = pipeline.fit(to_predict_mens_train[statistics_features], to_predict_mens_train["Outcome"])

    return statistics_model


### Inference

In [144]:
def inference(to_predict_mens_first_round_test, to_predict_mens_other_rounds_test, odds_model, statistics_model, features):

    pred_proba = odds_model.predict_proba(to_predict_mens_first_round_test[["final_odds"]].copy())[:,1]
    to_predict_mens_first_round_test["odds_pred"] = pred_proba

    pred_proba = statistics_model.predict_proba(to_predict_mens_first_round_test[features].copy())[:,1]
    to_predict_mens_first_round_test["statistics_pred"] = pred_proba

    to_predict_mens_first_round_test["Pred"] = (to_predict_mens_first_round_test.odds_pred * .75) + \
                                                (to_predict_mens_first_round_test.statistics_pred * .25)
    

   

    pred_proba = statistics_model.predict_proba(to_predict_mens_other_rounds_test[features].copy())[:,1]
    to_predict_mens_other_rounds_test["Pred"] = pred_proba
    to_predict_mens_other_rounds_test["statistics_pred"] = pred_proba

    mens_sub_tmp = pd.concat([to_predict_mens_first_round_test,
            to_predict_mens_other_rounds_test], axis=0)
    
    mens_sub = mens_sub_tmp[["ID", "Pred"]]
    
    return mens_sub, mens_sub_tmp



### Full Pipeline

In [145]:
statistics_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank']

season = 2025 

# get data
to_predict_mens_first_round_train, to_predict_mens_train, to_predict_mens_first_round_test, to_predict_mens_other_rounds_test,to_predict_mens_missing_games  = prepare_data(to_predict_mens, season)

# train
odds_model = train_odds_model(to_predict_mens_first_round_train)
statistics_model = train_statistics_model(to_predict_mens_train, statistics_features)

# inference
mens_sub, mens_sub_tmp = inference(to_predict_mens_first_round_test, to_predict_mens_other_rounds_test, odds_model, statistics_model, statistics_features) 

    

In [147]:
mens_sub_actual_not_games = to_predict_mens_missing_games[["ID", "Pred"]]

mens_sub.to_csv("mens_sub_actual_games_old.csv", index=False)

mens_sub = pd.concat([mens_sub,
        mens_sub_actual_not_games], axis=0)


### Womens

In [148]:
to_predict_women = pd.read_csv("to_predict_women.csv")

to_predict_women_train = to_predict_women[to_predict_women.Season != 2025] 

to_predict_women_test = to_predict_women[
                                        (to_predict_women.type == 'Prediction')  
                                        &  (to_predict_women.seed_diff.notnull())                                         
                                         ] 

to_predict_women_non_games = to_predict_women[
                                        (to_predict_women.type == 'Prediction')
                                        & (to_predict_women.seed_diff.isna())                                                 
                                         ] 



In [149]:
statistics_features = ['seed_diff', 't1_adj_margin', 't2_adj_margin']
best_params = {"C": .1}
model = LogisticRegression(**best_params)
pipeline = make_pipeline(StandardScaler(), model)
statistics_model = pipeline.fit(to_predict_women_train[statistics_features], to_predict_women_train["Outcome"])


In [150]:
pred_proba = statistics_model.predict_proba(to_predict_women_test[statistics_features].copy())[:,1]
to_predict_women_test["Pred"] = pred_proba
womens_sub = to_predict_women_test[["ID", "Pred"]]


/var/folders/h5/f91pbgmj0rj6v0ls3y8zc5l40000gn/T/ipykernel_73042/593629811.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  to_predict_women_test["Pred"] = pred_proba


In [151]:
womens_sub = pd.concat([womens_sub,
        to_predict_women_non_games[["ID", "Pred"]]], axis=0)


In [152]:
final_sub = pd.concat([mens_sub, womens_sub], axis=0)

In [153]:
len(final_sub)

131407

In [154]:
final_sub.to_csv("submission_3_19_25_old_approach.csv", index=False)

### Sanity Check

In [93]:
mens_sub_tmp[mens_sub_tmp["t2_TeamName"] == "St John's"][["Season", "Pred", "Outcome", 't1_TeamName', 't2_TeamName',
                                                            "Team1", "Team2"]].sort_values(by='Pred', ascending=False).head(5)


,Season,Pred,Outcome,t1_TeamName,t2_TeamName,Team1,Team2
9052,2025,0.809420,NaN,Auburn,St John's,1120,1385
27121,2025,0.797864,NaN,Duke,St John's,1181,1385
37612,2025,0.730086,NaN,Houston,St John's,1222,1385
4117,2025,0.680921,NaN,Alabama,St John's,1104,1385
31336,2025,0.679035,NaN,Florida,St John's,1196,1385


In [ ]:
# brier_score_loss(mens_sub_tmp["Outcome"], mens_sub_tmp["Pred"])

0.18875757835489718

In [ ]:
# # Round 1 correctness
# round1_correctness = mens_sub_tmp[mens_sub_tmp.GameRound == 1][["Pred", "Outcome", 't1_TeamName', 't2_TeamName', 
#                                                                 "t1_Seed", "t2_Seed","GameRound"]]
# round1_correctness["Outcome_Pred"] = np.where(round1_correctness.Pred > .5, 1, 0 )
# round1_correctness["Correct"] = np.where(round1_correctness["Outcome"] == round1_correctness["Outcome_Pred"], 1, 0)
# round1_correctness["Correct"].mean()

0.625

In [97]:
len(mens_sub_tmp)

2274

In [105]:
mens_sub_tmp[(mens_sub_tmp.t2_TeamName == "Wisconsin")][['t1_TeamName', 't2_TeamName', 'statistics_pred', 'odds_pred', 't1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank', 'final_odds', 'Pred']]

,t1_TeamName,t2_TeamName,statistics_pred,odds_pred,t1_adj_margin,t2_adj_margin,t1_final_rank,t2_final_rank,t1_OrdinalRank,t2_OrdinalRank,final_odds,Pred
51416,Montana,Wisconsin,0.112774,0.098069,3.397111,25.658121,72.613256,88.531436,25.0,25.0,16.5,0.101745
3827,Akron,Wisconsin,0.178952,NaN,7.413741,25.658121,77.776482,88.531436,25.0,25.0,16.5,0.178952
4187,Alabama,Wisconsin,0.708520,NaN,29.260605,25.658121,92.605689,88.531436,2.0,25.0,-22.5,0.708520
4904,Alabama St,Wisconsin,0.043716,NaN,-9.803138,25.658121,65.923737,88.531436,25.0,25.0,16.5,0.043716
5972,American Univ,Wisconsin,0.067123,NaN,-1.940634,25.658121,67.775784,88.531436,25.0,25.0,16.5,0.067123
...,...,...,...,...,...,...,...,...,...,...,...,...
67037,UCLA,Wisconsin,0.395506,NaN,19.683262,25.658121,86.033588,88.531436,25.0,25.0,-5.5,0.395506
67376,UNC Wilmington,Wisconsin,0.177413,NaN,7.944456,25.658121,77.337520,88.531436,25.0,25.0,16.5,0.177413
67679,Utah St,Wisconsin,0.286045,NaN,14.613800,25.658121,82.124635,88.531436,25.0,25.0,16.5,0.286045
67817,VCU,Wisconsin,0.451254,NaN,28.566672,25.658121,84.233153,88.531436,25.0,25.0,16.5,0.451254


In [39]:
mens_sub_tmp.columns

Index(['Unnamed: 0', 'type', 'ID', 'Pred', 'Season', 'Team1', 'Team2',
       'Outcome', 'Gender', 'margin', 't1_TeamName', 't1_FirstD1Season',
       't1_LastD1Season', 't2_TeamName', 't2_FirstD1Season', 't2_LastD1Season',
       'final_odds', 'GameRound', 't1_FGM', 't1_FGA', 't1_FGM3', 't1_FGA3',
       't1_OR', 't1_Ast', 't1_TO', 't1_Stl', 't1_PF', 't1_FTA', 't1_FTM',
       't1_PointDiff', 't2_FGM', 't2_FGA', 't2_FGM3', 't2_FGA3', 't2_OR',
       't2_Ast', 't2_TO', 't2_Stl', 't2_PF', 't2_FTA', 't2_FTM',
       't2_PointDiff', 't1_OrdinalRank', 't2_OrdinalRank', 't1_Seed',
       't2_Seed', 'seed_diff', 't1_adj_oe', 't1_adj_de', 't1_adj_margin',
       't2_adj_oe', 't2_adj_de', 't2_adj_margin', 't1_final_rank',
       't2_final_rank', 't1_top8_TO_stdev', 't1_top5_PRPG!_median',
       't1_top3_DR_median', 't1_top5_STL_cv', 't1_top3_Min%_median',
       't1_top8_TS_gini', 't1_top3_USG_gini', 't1_top8_BPM_weighted_mean',
       't2_top8_TO_stdev', 't2_top5_PRPG!_median', 't2_top3_DR_m